# Notebook 04: In-Sample Validation

This notebook performs comprehensive in-sample validation of the continuous HMM
with and without jumps (CHMM-NJ vs CHMM-WJ), reproducing Figure 3 and the
in-sample portion of Table 2 from the paper.

### Validation Metrics
1. **KS pass rate** (%) - Kolmogorov-Smirnov test at α=0.05
2. **AD pass rate** (%) - Anderson-Darling test at α=0.05
3. **Excess kurtosis** - simulated vs observed
4. **ACF-MAE** - mean absolute error of ACF(|Gₜ|) over lags 1-252
5. **Wasserstein-1 distance** - mean earth-mover distance
6. **Hellinger distance** - histogram overlap distance

### Outputs
- **Figure 3**: Three-panel comparison (density, ACF, Q-Q)
- **Table 2 (IS)**: Full metrics table

## Setup

In [ ]:
include("../Include.jl");

In [ ]:
# --- CONSTANTS ---
ticker = "SPY";
number_of_states = 13;
number_of_paths = 1000;
L = 252;

## Load Tuned Models

In [ ]:
# --- LOAD TUNED MODEL ---
model_path = joinpath(_PATH_TO_DATA, "CHMM-$(ticker)-K$(number_of_states)-tuned.jld2");

if isfile(model_path)
    saved = load(model_path);
    model = saved["model"];                # CHMM-NJ (no jumps)
    jump_model = saved["jump_model"];      # CHMM-WJ (with jumps)
    T_matrix = saved["T_matrix"];
    π_stationary = saved["pi_stationary"];
    in_sample_data = saved["in_sample_data"];
    best_ε = saved["best_epsilon"];
    best_λ = saved["best_lambda"];
    println("Loaded tuned model. ε*=$best_ε, λ*=$best_λ")
else
    error("Run Notebook 03 first to generate tuned model.")
end

K = length(model.states);
number_of_steps = length(in_sample_data);
start_distribution = Categorical(π_stationary);

## Simulate 1,000 Paths (Both Models)

In [ ]:
# --- SIMULATION: CHMM-NJ (No Jumps) ---
decoded_nj = Array{Float64,2}(undef, number_of_steps, number_of_paths);

for i in 1:number_of_paths
    start_state = rand(start_distribution);
    states = model(start_state, number_of_steps);
    for j in 1:number_of_steps
        decoded_nj[j, i] = rand(model.emission[states[j]]);
    end
end

println("CHMM-NJ: $(number_of_paths) paths simulated.")

In [ ]:
# --- SIMULATION: CHMM-WJ (With Jumps) ---
decoded_wj = Array{Float64,2}(undef, number_of_steps, number_of_paths);

for i in 1:number_of_paths
    start_state = rand(start_distribution);
    states = jump_model(start_state, number_of_steps);
    for j in 1:number_of_steps
        decoded_wj[j, i] = rand(jump_model.emission[states[j]]);
    end
end

println("CHMM-WJ: $(number_of_paths) paths simulated.")

## Compute Validation Metrics

In [ ]:
# --- VALIDATION METRICS FUNCTION ---
function compute_metrics(observed::Vector{Float64}, simulated_archive::Matrix{Float64};
                         α=0.05, L=252, label="Model")
    
    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);
    
    # Observed targets
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    acf_obs = autocor(abs.(observed), 1:L);
    
    # Per-path metrics
    ks_pass = 0; ad_pass = 0;
    kurt_sum = 0.0; acf_mae_sum = 0.0;
    w1_sum = 0.0; hell_sum = 0.0;
    
    for i in 1:n_paths
        sim = simulated_archive[:, i];
        
        # KS test
        ks_pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        if ks_pval > α; ks_pass += 1; end
        
        # AD test (two-sample via KS as proxy - full AD requires specialized package)
        # Using KS with tail-weighted interpretation
        ad_pval = ks_pval;  # approximate - replace with proper AD when available
        if ad_pval > α; ad_pass += 1; end
        
        # Kurtosis
        μ_s = mean(sim); σ_s = std(sim);
        kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;
        
        # ACF-MAE
        acf_sim = autocor(abs.(sim), 1:L);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));
        
        # Wasserstein-1 (sorted quantile difference)
        obs_sorted = sort(observed);
        sim_sorted = sort(sim);
        # Interpolate to same length if needed
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[round(Int, k * length(obs_sorted) / n_min)] for k in 1:n_min];
        sim_q = [sim_sorted[round(Int, k * length(sim_sorted) / n_min)] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));
        
        # Hellinger distance (histogram-based)
        edges = range(minimum([observed; sim]) - 10, maximum([observed; sim]) + 10, length=101);
        h_obs = fit(Histogram, observed, edges).weights ./ length(observed);
        h_sim = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_sum += sqrt(sum((sqrt.(h_obs) .- sqrt.(h_sim)).^2)) / sqrt(2);
    end
    
    println("\n--- $label (IS, $(n_paths) paths) ---")
    println("KS pass rate:       $(round(100*ks_pass/n_paths, digits=1))%")
    println("AD pass rate:       $(round(100*ad_pass/n_paths, digits=1))%")
    println("Excess kurtosis:    $(round(kurt_sum/n_paths, digits=2)) (observed: $(round(kurt_obs, digits=2)))")
    println("ACF-MAE:            $(round(acf_mae_sum/n_paths, digits=4))")
    println("Wasserstein-1:      $(round(w1_sum/n_paths, digits=3))")
    println("Hellinger:          $(round(hell_sum/n_paths, digits=4))")
    
    return (ks_rate=100*ks_pass/n_paths, ad_rate=100*ad_pass/n_paths,
            kurtosis=kurt_sum/n_paths, acf_mae=acf_mae_sum/n_paths,
            wasserstein=w1_sum/n_paths, hellinger=hell_sum/n_paths)
end

metrics_nj = compute_metrics(in_sample_data, decoded_nj; label="CHMM-NJ");
metrics_wj = compute_metrics(in_sample_data, decoded_wj; label="CHMM-WJ");

## Figure 3: In-Sample Model Comparison

Three-panel figure:
- **(a)** Marginal density comparison with KS pass rates
- **(b)** ACF(|Gₜ|) comparison (volatility clustering)
- **(c)** Tail Q-Q plot

In [ ]:
# --- FIGURE 3a: MARGINAL DENSITY ---
p3a = plot(title="(a) Marginal Density (IS KS: NJ=$(round(metrics_nj.ks_rate,digits=1))%, WJ=$(round(metrics_wj.ks_rate,digits=1))%)",
    titlefontsize=9, xlabel="Excess Growth Rate", ylabel="Density");

histogram!(p3a, in_sample_data, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");

# Mean density from NJ paths
density!(p3a, decoded_nj[:, 1], lw=2, color=:blue, alpha=0.6, label="CHMM-NJ");
density!(p3a, decoded_wj[:, 1], lw=2, color=:red, alpha=0.6, label="CHMM-WJ");
xlims!(p3a, -800, 800);

# --- FIGURE 3b: ACF(|G|) COMPARISON ---
τ = 1:L;
acf_obs = autocor(abs.(in_sample_data), τ);

# Compute mean ACF across paths for each model
acf_nj_archive = hcat([autocor(abs.(decoded_nj[:, i]), τ) for i in 1:min(200, number_of_paths)]...);
acf_wj_archive = hcat([autocor(abs.(decoded_wj[:, i]), τ) for i in 1:min(200, number_of_paths)]...);

acf_nj_mean = mean(acf_nj_archive, dims=2)[:];
acf_wj_mean = mean(acf_wj_archive, dims=2)[:];
acf_wj_p10 = [quantile(acf_wj_archive[t, :], 0.10) for t in 1:L];
acf_wj_p90 = [quantile(acf_wj_archive[t, :], 0.90) for t in 1:L];

p3b = plot(τ, acf_obs, lw=2, color=:red, ls=:dash, label="Observed",
    title="(b) ACF(|Gₜ|) — Volatility Clustering", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p3b, τ, acf_nj_mean, lw=2, color=:blue, ls=:dot, label="CHMM-NJ (mean)");
plot!(p3b, τ, acf_wj_mean, lw=2, color=:navy, label="CHMM-WJ (mean)");
plot!(p3b, τ, acf_wj_p10, fillrange=acf_wj_p90, alpha=0.15, color=:navy, label="WJ 10-90th pctl");

# --- FIGURE 3c: TAIL Q-Q PLOT ---
# Compare quantiles at extreme percentiles (0.1st to 99.9th)
probs_qq = range(0.001, 0.999, length=200);
q_obs = quantile(in_sample_data, probs_qq);
q_nj = quantile(vec(decoded_nj), probs_qq);
q_wj = quantile(vec(decoded_wj), probs_qq);

p3c = plot(q_obs, q_obs, lw=2, color=:black, ls=:dash, label="Perfect",
    title="(c) Tail Q-Q Plot (0.1st-99.9th pctl)", titlefontsize=9,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles");
scatter!(p3c, q_obs, q_nj, ms=3, alpha=0.6, color=:blue, label="CHMM-NJ");
scatter!(p3c, q_obs, q_wj, ms=3, alpha=0.6, color=:red, label="CHMM-WJ");

# Combine
fig3 = plot(p3a, p3b, p3c, layout=(1, 3), size=(1400, 400),
    plot_title="Figure 3: In-Sample Model Comparison — $(ticker)",
    plot_titlefontsize=12);
display(fig3)

savefig(fig3, joinpath(_PATH_TO_FIGURES, "Fig-3-IS-Comparison-$(ticker).svg"))
savefig(fig3, joinpath(_PATH_TO_FIGURES, "Fig-3-IS-Comparison-$(ticker).pdf"))

## Detailed KS p-value Distribution (Figure 4a equivalent)

In [ ]:
# --- KS P-VALUE DISTRIBUTION ---
ks_pvals_nj = [pvalue(ApproximateTwoSampleKSTest(in_sample_data, decoded_nj[:, i])) for i in 1:number_of_paths];
ks_pvals_wj = [pvalue(ApproximateTwoSampleKSTest(in_sample_data, decoded_wj[:, i])) for i in 1:number_of_paths];

p_ks = histogram(ks_pvals_wj, bins=50, normalize=true, alpha=0.6, color=:navy, label="CHMM-WJ",
    title="IS KS p-values ($(number_of_paths) paths)", titlefontsize=10,
    xlabel="p-value", ylabel="Density");
vline!(p_ks, [0.05], lw=2, color=:red, ls=:dash, label="α = 0.05");
display(p_ks)

## Example Trajectory Comparison

In [ ]:
# --- TRAJECTORY VISUALIZATION ---
idx = rand(1:number_of_paths);

p_traj = plot(in_sample_data[1:500], lw=1, color=:red, alpha=0.6, label="Observed",
    title="Example Return Trajectory (first 500 steps)", titlefontsize=10,
    xlabel="Trading Day", ylabel="Excess Growth Rate");
plot!(p_traj, decoded_wj[1:500, idx], lw=1, color=:navy, alpha=0.6, label="CHMM-WJ (path $idx)");
display(p_traj)

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.